# Task 1 - Step 0: Setup, configuration and experimental design

This notebook prepares everything the rest of Task 1 relies on:

1. installs / checks the libraries,
2. loads the shared configuration ([configs/task1_config.json](configs/task1_config.json)),
3. downloads STL-10,
4. **states the hypothesis and metric for every experimental-design choice** (the assignment requires this *before* interpreting any result).

## Notebook run order

| # | Notebook | Purpose | Main outputs |
|---|----------|---------|--------------|
| 00 | `00_setup_and_design` | environment, dataset, hypotheses | - |
| 01 | `01_make_subset` | class-balanced 500-image test subset + 80/20 head split | `results/subset/*.json`, `cache/sets/clean.pt` |
| 02 | `02_make_transforms` | grayscale, hue rotation, translation, patch shuffle | `cache/sets/*.pt` |
| 03 | `03_make_cue_conflicts` | AdaIN cue-conflict images + rejection rule | `cache/sets/cue_conflict.pt`, `results/cue_conflicts/` |
| 04 | `04_backbones_and_heads` | frozen features, linear heads, CLIP zero-shot | `cache/features/*.pt`, `cache/heads/` |
| 05 | `05_evaluate_bias` | Steps 1-5: baseline, colour, shape/texture, translation, patches | `results/tables`, `results/figures` |
| 06 | `06_representation_analysis` | Step 6: cosine stability + t-SNE + prediction-vs-representation | `results/tables`, `results/figures` |
| 07 | `07_export_results` | copy results to top-level `results/task1`, `figures/task1`; build `results.json` | `../results/task1/` |

Transformation generation (01-03) is deliberately separate from evaluation (04-06) so **exactly the same images** are fed to every model.

## 0.1 Install dependencies
Run once (uncomment). Needs `torch`, `torchvision`, `open_clip_torch`, `scikit-learn`, `pandas`, `matplotlib`, `scipy`.

In [1]:
# %pip install torch torchvision open_clip_torch scikit-learn pandas matplotlib scipy tqdm
# Optional (only if you prefer UMAP over t-SNE in notebook 06):
# %pip install umap-learn

## 0.2 Common header + library versions

In [2]:
# ---- Common header (identical in every Task 1 notebook) ----
import json, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# Locate the task1/ folder no matter where Jupyter was launched from.
ROOT = Path.cwd()
if not (ROOT / "configs").exists() and (ROOT / "task1" / "configs").exists():
    ROOT = ROOT / "task1"

CFG = json.loads((ROOT / "configs" / "task1_config.json").read_text())
SEED = CFG["seed"]

# Small, git-tracked outputs -> results/.  Large tensors/weights/raw data -> cache/ and data/ (git-ignored).
RES = ROOT / "results"
CACHE = ROOT / "cache"
DATA = ROOT / "data"
for p in [RES / "subset", RES / "cue_conflicts", RES / "tables", RES / "figures",
          CACHE / "sets", CACHE / "features", CACHE / "heads", CACHE / "weights", DATA]:
    p.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def set_seed(seed=SEED):
    """Fix every RNG we use so reruns reproduce the same numbers."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed()
print("ROOT:", ROOT, "| device:", DEVICE, "| seed:", SEED)

ROOT: c:\Users\afifh\Desktop\ATML\PA1\task1 | device: cuda | seed: 6304


In [3]:
import torchvision, sklearn
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| sklearn", sklearn.__version__)
try:
    import open_clip
    print("open_clip", open_clip.__version__)
except ImportError:
    print("open_clip missing -> run the %pip cell above")
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

torch 2.11.0+cu128 | torchvision 0.26.0+cu128 | sklearn 1.9.1
open_clip 3.3.0
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


## 0.3 Configuration

All hyper-parameters live in one JSON file so every notebook uses identical settings. Printing it here also documents the exact configuration behind the reported numbers.

In [4]:
print(json.dumps(CFG, indent=2))

{
  "seed": 6304,
  "dataset": "STL10",
  "img_size": 224,
  "n_test_subset": 500,
  "head": {
    "max_epochs": 50,
    "lr": 0.001,
    "weight_decay": 0.0001,
    "patience": 5,
    "batch_size": 256,
    "val_frac": 0.2
  },
  "clip": {
    "model": "ViT-B-32",
    "pretrained": "openai",
    "prompt": "a photo of a {}."
  },
  "color": {
    "hue_shift": 0.5
  },
  "translation": {
    "displacements": [
      8,
      16,
      32
    ],
    "directions": [
      "right",
      "left",
      "down",
      "up"
    ]
  },
  "patch": {
    "grid": 4
  },
  "cue_conflict": {
    "pairs": [
      [
        "airplane",
        "horse"
      ],
      [
        "bird",
        "car"
      ],
      [
        "cat",
        "ship"
      ],
      [
        "deer",
        "truck"
      ],
      [
        "dog",
        "monkey"
      ]
    ],
    "n_generate_per_cell": 40,
    "n_keep_per_cell": 25,
    "style_alpha": 1.0,
    "reject": {
      "min_edge_corr": 0.2,
      "min_gray_std": 0

## 0.4 Download STL-10

STL-10: 10 classes, 96x96 RGB images, 5000 official training images (500/class) and 8000 official test images (800/class). We use the official train split for the linear heads (80/20 stratified train/val) and a class-balanced 500-image subset of the official test split for all interventions.

In [5]:
from torchvision.datasets import STL10

train_ds = STL10(root=str(DATA), split="train", download=True)
test_ds = STL10(root=str(DATA), split="test", download=True)
CLASSES = train_ds.classes
print("classes:", CLASSES)
print("train:", len(train_ds), "| test:", len(test_ds), "| image size:", train_ds[0][0].size)

# Sanity: class balance of the official splits
print("train counts:", np.bincount(train_ds.labels))
print("test counts :", np.bincount(test_ds.labels))

classes: ['airplane', 'bird', 'car', 'cat', 'deer', 'dog', 'horse', 'monkey', 'ship', 'truck']
train: 5000 | test: 8000 | image size: (96, 96)
train counts: [500 500 500 500 500 500 500 500 500 500]
test counts : [800 800 800 800 800 800 800 800 800 800]


## 0.5 Experimental-design choices, hypotheses and metrics

The assignment lets us choose four things. Each is fixed **before** looking at results.

### (a) Dataset - STL-10
*Why:* ten distinct object classes, low computational cost, recommended by the assignment. 96x96 images are bicubically upsampled to 224x224 (the common intervention resolution).
*Confound to remember:* upsampling makes images smooth, so high-frequency texture is weaker than in native-resolution ImageNet images.

### (b) Cue-conflict class pairs and style strength
Pairs (shape/content class <-> texture/style class), each used in both directions:
`airplane/horse, bird/car, cat/ship, deer/truck, dog/monkey`.
Pairs mix animals and vehicles with visually different silhouettes so that a stylised image is unlikely to be ambiguous; `cat/ship` and `dog/monkey` also include fur-vs-smooth contrasts. AdaIN style strength alpha = 1.0 (full stylisation).

| | |
|---|---|
| **Hypothesis** | With full stylisation, the ImageNet-supervised ResNet-50 will assign more decisions to the *texture* label than the ViT-B/16 and CLIP do (shape bias: CLIP >= ViT > ResNet), but coverage will be well below 100 % for all models because AdaIN also destroys some shape. |
| **Metric** | Shape Bias = N_shape/(N_shape+N_texture); Coverage = (N_shape+N_texture)/N_total; reported together with the count of "other" decisions. |

### (c) Additional colour intervention - fixed hue rotation (+0.5, i.e. 180 degrees)
Hue rotation keeps luminance/edges/geometry and preserves the *set* of pixel saturations and values but re-maps every colour to its complement. Grayscale tests sensitivity to *removing* chromatic information; hue rotation tests sensitivity to *changing* it while keeping colour statistics.

| | |
|---|---|
| **Hypothesis** | Grayscale will hurt little (< 5 pts accuracy) for all models on STL-10. Hue rotation will hurt more than grayscale for models that use colour as a *class prior* (sky/water blue, grass green), most likely ResNet-50; CLIP is expected to be the most robust. |
| **Metric** | Delta top-1 accuracy vs the model's own clean accuracy, and prediction consistency with clean. |

### (d) Representation-visualisation - t-SNE
Fitted separately per backbone on the concatenation of clean + transformed features (cosine metric, perplexity 30, 1000 iterations, seed 6304). Chosen because we mainly care about local-neighbourhood structure (do transformed images leave their clean cluster?).

| | |
|---|---|
| **Hypothesis** | Patch-shuffled and cue-conflict features will move visibly away from their clean class clusters (largest shift for the ResNet), while grayscale and small translations stay near their clean counterparts. |
| **Metric** | Cosine stability I_T (Step 6), plus visual inspection of class separation / clean-vs-transformed mixing. Absolute t-SNE coordinates are **never** compared across backbones. |

### (e) Other hypotheses (translation, patch structure)
* **Translation** - accuracy and consistency decline with displacement; the ViT and CLIP (32-px patches, positional embeddings) are expected to be *more* sensitive at displacements that are not a multiple of the patch size (8/16 px) than the ResNet's global-average-pooled features. Metric: accuracy and Consistency(delta) vs displacement.
* **Patch shuffle (4x4)** - a big accuracy drop for all models; a shape-reliant model drops most. A *confident* wrong prediction is possible, so mean confidence and the predicted-class histogram are reported next to the accuracy drop.

In [6]:
# Persist the design decisions next to the results so they are timestamped with the code that ran.
design = {
    "dataset": "STL10",
    "cue_conflict_pairs": CFG["cue_conflict"]["pairs"],
    "style_alpha": CFG["cue_conflict"]["style_alpha"],
    "extra_color_intervention": f"hue_rotation(+{CFG['color']['hue_shift']})",
    "visualization": "t-SNE",
    "tsne": CFG["tsne"],
    "seed": SEED,
}
(RES / "design_choices.json").write_text(json.dumps(design, indent=2))
print("saved", RES / "design_choices.json")

saved c:\Users\afifh\Desktop\ATML\PA1\task1\results\design_choices.json
